# Proyecto Final — Telecomunicaciones: identificar operadores ineficaces

Notebook reproducible para limpieza, EDA, métricas, hipótesis y dashboards.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

clients = pd.read_csv('telecom_clients.csv')
calls = pd.read_csv('telecom_dataset_new.csv')
calls['date'] = pd.to_datetime(calls['date'])
calls['date_day'] = calls['date'].dt.date
calls.head()

## 1. Limpieza

In [ ]:
print('Duplicados:', calls.duplicated().sum())
print('Operador faltante:', calls['operator_id'].isna().sum())

calls = calls.drop_duplicates().dropna(subset=['operator_id']).copy()
calls['operator_id'] = calls['operator_id'].astype(int)

bad = calls['is_missed_call'] & (calls['call_duration'] > 0)
print('Inconsistencias corregidas:', bad.sum())
calls.loc[bad, 'is_missed_call'] = False
calls['wait_duration'] = calls['total_call_duration'] - calls['call_duration']

op_day = calls.groupby(['operator_id','date_day']).agg(total_sec=('total_call_duration','sum')).reset_index()
anomalous_ops = op_day.loc[op_day['total_sec'] > 12*60*60, 'operator_id'].unique()
calls = calls[~calls['operator_id'].isin(anomalous_ops)].copy()

calls = calls.merge(clients[['user_id','tariff_plan','date_start']], on='user_id', how='left')

## 2. EDA

In [ ]:
incoming = calls[calls['direction']=='in']
outgoing = calls[calls['direction']=='out']
print('Llamadas totales:', calls['calls_count'].sum())
print('Entrantes:', incoming['calls_count'].sum())
print('Salientes:', outgoing['calls_count'].sum())
print('Tasa pérdida entrante:', incoming.loc[incoming['is_missed_call'],'calls_count'].sum()/incoming['calls_count'].sum())
print('Espera media entrante:', incoming['wait_duration'].sum()/incoming['calls_count'].sum())

## 3. Métricas por operador

In [ ]:
op_in = incoming.groupby('operator_id').agg(
    incoming_calls=('calls_count','sum'),
    missed_calls=('calls_count', lambda s: s[incoming.loc[s.index,'is_missed_call']].sum()),
    wait_seconds=('wait_duration','sum')
)
op_in['missed_rate'] = op_in['missed_calls']/op_in['incoming_calls']
op_in['avg_wait_sec'] = op_in['wait_seconds']/op_in['incoming_calls']
op_out = outgoing.groupby('operator_id').agg(outgoing_calls=('calls_count','sum'))
metrics = op_in.join(op_out, how='outer').fillna(0).reset_index()
metrics = metrics.merge(calls[['operator_id','tariff_plan']].drop_duplicates(), on='operator_id', how='left')

active = metrics[metrics['incoming_calls'] >= 20]
missed_threshold = active['missed_rate'].quantile(.75)
wait_threshold = active['avg_wait_sec'].quantile(.75)
out_threshold = active.loc[active['outgoing_calls'] > 0, 'outgoing_calls'].quantile(.25)

metrics['high_missed_rate'] = metrics['missed_rate'] >= missed_threshold
metrics['high_wait'] = metrics['avg_wait_sec'] >= wait_threshold
metrics['low_outbound'] = metrics['outgoing_calls'] <= out_threshold
metrics['incoming_risk'] = (
    (metrics['incoming_calls'] >= 20) &
    metrics['high_missed_rate'] &
    metrics['high_wait']
)
risk = metrics[metrics['incoming_risk']].sort_values(['missed_rate','avg_wait_sec'], ascending=False)
risk.head(10)

## 4. Hipótesis estadísticas

In [ ]:
test = metrics[metrics['incoming_calls'] >= 20]

# H0: la espera media entrante tiene la misma distribución en A, B y C
wait_groups = [g['avg_wait_sec'].values for _, g in test.groupby('tariff_plan')]
print('Kruskal-Wallis espera:', stats.kruskal(*wait_groups))

# H0: la tasa de llamadas perdidas tiene la misma distribución en A, B y C
missed_groups = [g['missed_rate'].values for _, g in test.groupby('tariff_plan')]
print('Kruskal-Wallis tasa de pérdida:', stats.kruskal(*missed_groups))

# Comparaciones post hoc de espera
for a,b in combinations(['A','B','C'],2):
    x = test.loc[test['tariff_plan']==a,'avg_wait_sec']
    y = test.loc[test['tariff_plan']==b,'avg_wait_sec']
    print(a,b,stats.mannwhitneyu(x,y,alternative='two-sided'))

## 5. Dashboards

Usa `telecom_dashboard_data.csv` en Tableau.

**Sugerencia 1:** histograma de `call_duration`, pie de `call_type`, filtro `direction`.

**Sugerencia 2:** histograma de `calls_count` agregado por día, pie de `call_type`, filtro `internal`.